---
# Part 1 - Results from the GS's are Analyzed 
---
- purpose of the Part 1 of thenotebook is to analyze the metrics from an initial exploratory gridsearch 
- after this largetr GS, i run a more concentrated GS with less parameters , parameters are eliminated based on the visual analysis below 
- more robust methods may be used but this works for now 

In [6]:
import pandas as pd
import pickle
import random
import numpy as np
import os
import itertools
from joblib import Parallel, delayed , parallel_backend
from collections import defaultdict
import math
import torch.nn as nn
import json

from Equations_Run_Combo_V_2 import *



import pickle
with open('/Users/cs/Desktop/LSTM_ETF/short_dfs.pkl', 'rb') as f:
    loaded_dfs = pickle.load(f)

with open("/Users/cs/Desktop/LSTM_ETF/lagged_cache.pkl", "rb") as f:
    lagged_cache = pickle.load(f)


#/home/charifslmn/

---
# Part 1 - GS Results Exploration
---

In [7]:

for i in range(1):  
    with open(f'/Users/cs/Desktop/GS_21_01_to_22_12_chunk_1_32_10percentPOS_Vset_HOD.json', 'r') as f:
        results = json.load(f)




In [8]:
import json
import numpy as np
import copy


scored_results = []
for entry in results:
    avg = entry['cv_sets'].get('overall_metrics', {})
    acc = avg.get('accuracy') or 0.0
    prec_up = avg.get('precision_up') or 0.0
    recall_up = avg.get('recall_up') or 0.0

    total_score = acc  # or add prec_up, etc.
    
    if recall_up > 10 and prec_up > 65:

        scored_results.append((total_score, entry))

# Step 2: Sort by score descending
scored_results.sort(reverse=True, key=lambda x: x[0])


top = scored_results[:70]

for i, j in top:
    print(j["combo_number"], '<--->' , j["cv_sets"]["overall_metrics"] , '<---->' , j["parameters"] )

# Step 4: Extract the parameter combos
combos = [j["parameters"] for i, j in top]

print(f"Total Combos Selected: {len(top)}")

829 <---> {'accuracy': 91.66666666666666, 'precision_up': 100.0, 'recall_up': 50.0, 'precision_down': 90.9090909090909, 'recall_down': 100.0} <----> {'binary_0_1_cutoff_ret_rate_percentage': 0.1, 'learning_rate': 0.05, 'num_epochs': 70, 'batch_size': 50, 'use_bidirectional': False, 'lag': 8, 'input_size': 12, 'hidden_size': 55, 'num_layers': 2, 'use_monthly_dfs_only': True, 'use_binary_0_1_retRate': False, 'use_custom_loss_function_BCE_THRESH': False, 'use_custom_loss_function_BCE_THRESH_AND_SEVERITY': False, 'use_LOW_weights_for_BCE_custom_loss': True, 'pred_threshold_sigmoid01_up': None, 'use_binary_neg1_1': False, 'use_ret_rate': False, 'use_print_acc': False, 'use_dropout': False, 'use_class_weighting': True, 'is_deterministic': True, 'seed_num': 42, 'use_existing_lagged_data': True, 'use_dynamic_weights': False, 'use_binary_0_1_retRate_custom_neg': False, 'use_binary_0_1_retRate_custom_pos': True, 'POS_weight_multiplier': 1.5, 'use_rolling_fixed_train_size': False, 'use_existing_i

In [9]:
### pickle combos 

with open('/Users/cs/Desktop/LSTM_ETF/DIST_DISC_comboc_GS_21_01_to_22_12_chunk_1_32_10percentPOS_Vset_HOD.pkl', 'wb') as f:
    pickle.dump(combos, f)

In [ ]:
from Equations_Run_Combo_V_2 import *
from Equations_Ensembles_Dist import *

In [11]:



# cc = {
#     "use_USO_wticoncat_predictor_WEEKLY_END_MO" : False , "use_UCO_wticoncat_predictor_WEEKLY_END_MO" : True ,
#     "use_HUC_wticoncat_predictor_WEEKLY_END_MO" : False , "use_HOD_wticoncat_predictor_WEEKLY_END_MO" : False ,
#     "use_CRUD_wticoncat_predictor_WEEKLY_END_MO" : False , "use_SCO_wticoncat_predictor_WEEKLY_END_MO" : False ,

#         "learning_rate": 0.001, "num_epochs": 10,
#         "batch_size": 32, "use_bidirectional": False,
#         "lag": 2, "input_size": 12,
#         "hidden_size": 20, "num_layers": 2,

#     "use_monthly_dfs_only": True,

# "use_binary_0_1_retRate": False,

#         "use_binary_neg1_1": False,
# "use_ret_rate": False,
#         "use_print_acc": False,"use_dropout": False,
#         # iter_per_valSET: int,
#                                                         "use_class_weighting": True, 
#         "is_deterministic": True,"seed_num": 42,

#         "use_existing_lagged_data": True, "use_dynamic_weights": False    ,

#         "use_binary_0_1_retRate_custom_neg": False,
#         "use_binary_0_1_retRate_custom_pos": True,


#                     "binary_0_1_cutoff_ret_rate_percentage": 0.1,  ### cutoff for the  use_binary_0_1_retRate_custom_pos ot use_binary_0_1_retRate_custom_neg

#         "POS_weight_multiplier": 1,

#         "use_rolling_fixed_train_size": False, "use_existing_initial_weights": False,"state_dict": None,

#                 "use_custom_loss_function_BCE_THRESH": False, # NEW NEW NE
#                 "use_custom_loss_function_BCE_THRESH_AND_SEVERITY": False, # NEW NEW NEW
#                 "use_LOW_weights_for_BCE_custom_loss": False, # NEW NEW NEW
#                 "pred_threshold_sigmoid01_up": None ,
                
                
            
#         'train_start_month'  : "2004-01",
#         'val_start_month'  : '2020-01' ,    # test_start_month = '2022-01' ; test_end_month = '2022-12' 
#         'val_end_month'  : '2021-12' ,
#         'num_preds_per_fold' : 3} # NEW NEW NEW



# res = distribution_discovery(cc, combo_index = 0 , number_of_seeds = 4)

#### ----------------------                 RUN COMBOS
with parallel_backend("loky", n_jobs=3):
    results = Parallel()(
        delayed(distribution_discovery)(c, combo_index = i , number_of_seeds = 70)
        
        for i, c in enumerate(combos)
    )




****************** STARTED RUN FOR COMBO INDEX 0
****************** STARTED RUN FOR COMBO INDEX 2
****************** STARTED RUN FOR COMBO INDEX 1


Fatal Python error: init_import_site: Failed to import the site module
Python runtime state: initialized
Fatal Python error: init_import_site: Failed to import the site module
Python runtime state: initialized
Fatal Python error: init_import_site: Failed to import the site module
Python runtime state: initialized
Fatal Python error: init_import_site: Failed to import the site module
Python runtime state: initialized
Fatal Python error: init_import_site: Failed to import the site module
Python runtime state: initialized
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1176, in exec_module
  File "<frozen site>", line 716, in <module>
  File "<frozen site>", line 699, in main
  File "<frozen site>", line 631, in venv
  File "<frozen site>", line 

KeyboardInterrupt: 

In [ ]:

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, numpy.integer):
            return int(obj)
        elif isinstance(obj, numpy.ndarray):
            return obj.tolist()
        elif isinstance(obj, pd.DatetimeIndex):
            return obj.tolist()
        elif isinstance(obj, pd.Timestamp):
            return obj.isoformat()
        else:
            return super(NpEncoder, self).default(obj)


file = f"70_models_GS_21_01_to_22_12_DIST_Discovery_10percentPOS_Vset_HOD.json"

with open(file, "w") as f_json:
    json.dump(results, f_json, cls=NpEncoder, indent=2)